# Notebook 04: Simulator-First Benchmark

Prerequisite: read Chapter 04 first for the benchmark rules and thresholds.

This lab is practice-first. You will build a compact benchmark report after the chapter concept is clear.

## Cell-by-cell Guide
Read this first so every cell is easy to follow.
1. Understand concept and expected result - Notebook 04: Simulator-First Benchmark
2. Understand concept and expected result - Real-World Scenario
3. Run logic and inspect output - from qiskit import QuantumCircuit
4. Run logic and inspect output - expected = {
5. Run logic and inspect output - passes = (report["decision"] == "PASS").sum()
6. Understand concept and expected result - Optional extension

## Real-World Scenario
**Scenario:** Pilot Benchmark Gate Before Hardware

**Problem:** A team wants to avoid sending weak candidates to expensive hardware runs.
**Baseline:** Require simulator benchmark pass table for identity, X, and H circuits.
**Metric to watch:** Pass count, max deviation, and decision consistency.
**Practical takeaway:** Only promote candidates that pass controlled simulator thresholds.

## Imports and Purpose
This lab uses `QuantumCircuit` to build benchmark circuits, `AerSimulator` to run them locally, and `pandas` to assemble the comparison report.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import pandas as pd

sim = AerSimulator()
shots = 1024

circuits = {}

qc_identity = QuantumCircuit(1, 1)
qc_identity.measure(0, 0)
circuits["identity"] = qc_identity

qc_x = QuantumCircuit(1, 1)
qc_x.x(0)
qc_x.measure(0, 0)
circuits["x_flip"] = qc_x

qc_h = QuantumCircuit(1, 1)
qc_h.h(0)
qc_h.measure(0, 0)
circuits["h_superposition"] = qc_h

In [ ]:
expected = {
    "identity": {"0": 1.0, "1": 0.0},
    "x_flip": {"0": 0.0, "1": 1.0},
    "h_superposition": {"0": 0.5, "1": 0.5},
}

thresholds = {
    "identity": 0.98,
    "x_flip": 0.98,
    "h_superposition": 0.10,
}

report_rows = []
for name, qc in circuits.items():
    counts = sim.run(qc, shots=shots).result().get_counts()
    p0 = counts.get("0", 0) / shots
    p1 = counts.get("1", 0) / shots

    if name in ["identity", "x_flip"]:
        target = "p0" if name == "identity" else "p1"
        value = p0 if target == "p0" else p1
        decision = "PASS" if value >= thresholds[name] else "INVESTIGATE"
        metric = f"{target} >= {thresholds[name]}"
    else:
        diff = abs(p0 - p1)
        decision = "PASS" if diff <= thresholds[name] else "INVESTIGATE"
        metric = f"|p0-p1| <= {thresholds[name]}"

    report_rows.append({
        "circuit": name,
        "depth": qc.depth(),
        "shots": shots,
        "p0": round(p0, 4),
        "p1": round(p1, 4),
        "rule": metric,
        "decision": decision
    })

report = pd.DataFrame(report_rows)
report

In [ ]:
passes = (report["decision"] == "PASS").sum()
total = len(report)
print(f"Benchmark summary: {passes}/{total} circuits passed")
if passes < total:
    print("Action: inspect circuit setup, shot count, and simulator configuration.")
else:
    print("Action: proceed to noise and fidelity chapters.")

## Optional extension
Repeat the report across 5 runs and add columns for mean and spread per metric.